## An evolution strategy

CMA-ES against TPE on the same two dimensions at an equal budget. It also carries the
plateau stop that TPE does not: `no_improvement` ends a search that has stopped learning,
which a fixed batch budget cannot see.

**What to look for:** the same axes as the TPE notebook, so the two curves are directly
comparable. A search that flattens early and keeps spending is the failure this stop
exists to prevent.


In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import json, os, sqlite3
import pandas as pd
import matplotlib.pyplot as plt

def load_units(data_dir):
    """One row per evaluated cell: its parameters, objectives and measures.

    Read from campaign.db rather than data.db because that is where a SEARCH records what
    it scored -- data.db holds per-run tables, and a search's unit of analysis is the cell.
    """
    db = os.path.join(data_dir, 'campaign.db')
    if not os.path.exists(db):
        return pd.DataFrame()
    with sqlite3.connect(db) as conn:
        units = pd.read_sql_query(
            "SELECT u.paramset_id, u.config_name, u.params_json, u.objectives_json,"
            "       u.measures_json, u.n_samples, u.status, b.idx AS batch"
            "  FROM unit u LEFT JOIN batch b ON b.id = u.batch_id"
            " ORDER BY b.idx, u.id", conn)
    if units.empty:
        return units
    for col, prefix in (('params_json', ''), ('objectives_json', ''), ('measures_json', 'm_')):
        expanded = units[col].apply(lambda s: json.loads(s) if s else {}).apply(pd.Series)
        expanded.columns = [f'{prefix}{c}' for c in expanded.columns]
        units = pd.concat([units.drop(columns=[col]), expanded], axis=1)
    return units

units = load_units(DATA_DIR)
scored = units[units['status'] == 'evaluated'] if 'status' in units else units
print(f"{len(units)} cell(s) recorded, {len(scored)} scored")

if scored.empty:
    print("No scored cells yet. A search records a cell once its batch has been evaluated;"
          "\nif this campaign failed early, its controller log says why.")


In [ ]:
TITLE = 'CMA-ES — a different way down'

# Best-so-far against evaluations. The question is not what it found but how fast, so the
# x axis is spend and the line is the only thing that matters.
if not scored.empty:
    order = scored.reset_index(drop=True)
    best_so_far = order['robustness'].cummin()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(range(1, len(order) + 1), best_so_far, drawstyle='steps-post', linewidth=2)
    ax.scatter(range(1, len(order) + 1), order['robustness'], s=18, alpha=0.45,
               label='each cell')
    ax.axhline(0, color='grey', linestyle='--', linewidth=1, label='failure boundary')
    ax.set_xlabel('cells evaluated'); ax.set_ylabel('robustness')
    ax.set_title('%s: worst found so far' % TITLE)
    ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
if not scored.empty:
    failed = (scored['robustness'] < 0).sum()
    print(f"cells scored          : {len(scored)}")
    print(f"runs spent            : {int(scored['n_samples'].sum())}")
    print(f"cells that failed     : {failed}  ({failed / len(scored):.0%})")
    print(f"worst robustness      : {scored['robustness'].min():.3f}")
    print()
    order = scored.reset_index(drop=True)
    best = order['robustness'].cummin()
    final = best.iloc[-1]
    print("What this campaign is FOR -- how FEW evaluations found it:")
    print(f"  best crossing found : {final:.3f}")
    for frac in (0.5, 0.9, 1.0):
        target = final * frac
        hit = (best <= target).idxmax() + 1 if (best <= target).any() else None
        label = f"{frac:.0%} of that depth"
        print(f"  {label:<20}: {hit} evaluation(s)" if hit else f"  {label:<20}: not reached")
    print(f"  evaluations spent   : {len(order)}")
    print("  Read against the other sampler at the same runs: budget. Fewer evaluations to")
    print("  the same depth is the whole claim; the final number alone is not.")
